# FitTrack Product Analytics: Exploratory Notebook
This is an interactive walkthrough of the analysis. The reusable logic lives in `../src` and `../sql`, and this notebook explores it step by step.

**Business question:** Where are we losing users between signup and paid, and which levers (onboarding, features, channels) will grow subscription revenue?

In [1]:
import sys, sqlite3
sys.path.append('../src')
import pandas as pd
from scipy import stats
from build_database import build
from analysis import two_prop_ztest, srm_check
con = sqlite3.connect(build())
sql = lambda f: pd.read_sql(open(f'../sql/{f}').read(), con)
pd.set_option('display.width', 160)

## 1. Data sanity checks

In [2]:
for t in ['users', 'events', 'subscriptions']:
    print(t, pd.read_sql(f'SELECT COUNT(*) n FROM {t}', con).n[0])
pd.read_sql('SELECT event_name, COUNT(*) n FROM events GROUP BY 1 ORDER BY n DESC', con)

users 25000
events 124228
subscriptions 1477


,event_name,n
0,workout_completed,69017
1,signup,25000
2,onboarding_complete,15584
3,feature_used,9299
4,trial_started,3563
5,subscription_started,1477
6,subscription_cancelled,288


In [3]:
# Every user should have exactly one signup event, and nobody should subscribe without a trial
pd.read_sql('''SELECT
  (SELECT COUNT(*) FROM users) - (SELECT COUNT(DISTINCT user_id) FROM events WHERE event_name='signup') AS missing_signups,
  (SELECT COUNT(*) FROM subscriptions s WHERE NOT EXISTS (SELECT 1 FROM events e WHERE e.user_id=s.user_id AND e.event_name='trial_started')) AS subs_without_trial''', con)

,missing_signups,subs_without_trial
0,0,0


## 2. Headline KPIs

In [4]:
sql('01_kpi_overview.sql')

,total_users,onboarding_rate_pct,activation_rate_pct,paid_conversion_pct,active_mrr_usd
0,25000,62.3,37.4,5.91,13548.49


## 3. Funnel by device
The onboard → first workout step is the largest drop-off, and Android trails iOS by about 14 pp on it.

In [5]:
sql('02_funnel.sql')

,device,signups,onboarded,first_workout,trial,paid,signup_to_onboard_pct,onboard_to_workout_pct,workout_to_trial_pct,trial_to_paid_pct
0,all,25000,15584,9346,3563,1477,62.3,60.0,38.1,41.5
1,ios,11976,7519,4940,1881,793,62.8,65.7,38.1,42.2
2,android,10019,6176,3180,1202,501,61.6,51.5,37.8,41.7
3,web,3005,1889,1226,480,183,62.9,64.9,39.2,38.1


## 4. Onboarding A/B test
The test checks sample ratio mismatch first, then runs two-proportion z-tests on the primary and secondary metrics.

In [6]:
ab = sql('04_ab_test_onboarding.sql').set_index('experiment_group')
print('SRM p-value:', round(srm_check(ab.loc['control','users'], ab.loc['treatment','users']), 3))
rows = []
for m in ['onboarded', 'activated', 'trials']:
    p1, p2, z, p, ci = two_prop_ztest(ab.loc['control', m], ab.loc['control','users'], ab.loc['treatment', m], ab.loc['treatment','users'])
    rows.append(dict(metric=m, control=round(p1,4), treatment=round(p2,4), lift_pp=round((p2-p1)*100,2), ci_pp=(round(ci[0]*100,2), round(ci[1]*100,2)), p_value=p))
pd.DataFrame(rows)

SRM p-value: 0.982


,metric,control,treatment,lift_pp,ci_pp,p_value
0,onboarded,0.6081,0.6877,7.96,"(5.86, 10.05)",1.207923e-13
1,activated,0.3745,0.4176,4.31,"(2.16, 6.46)",8.742459e-05
2,trials,0.1514,0.1623,1.09,"(-0.51, 2.69)",1.809111e-01


In [7]:
# Power check: minimum detectable effect for onboarding completion at 80% power, alpha = 0.05
import numpy as np
n = ab.loc['control','users']; p = ab.loc['control','onboarded'] / n
mde = (stats.norm.ppf(0.975) + stats.norm.ppf(0.8)) * np.sqrt(2 * p * (1 - p) / n)
print(f'MDE ≈ {mde*100:.2f} pp with {n:,} users per arm')

MDE ≈ 3.07 pp with 3,963 users per arm


## 5. Feature adoption vs. retention

In [8]:
sql('05_feature_adoption_retention.sql')

,segment,users,week4_retention_pct,paid_conversion_pct
0,adopted_workout_plans,2482,40.2,22.6
1,did_not_adopt,4971,19.7,14.5


## 6. Channel quality

In [9]:
sql('06_channel_performance.sql')

,acquisition_channel,signups,activation_pct,paid_conversion_pct,revenue_to_date_usd,revenue_per_signup_usd
0,referral,2527,51.2,12.62,8793.0,3.48
1,organic,7448,41.2,7.14,14550.0,1.95
2,app_store,3742,38.0,6.12,6111.0,1.63
3,paid_search,5055,36.0,4.99,6588.0,1.30
4,paid_social,6228,28.0,2.33,4022.0,0.65


## 7. Cohort retention

In [10]:
ret = sql('03_weekly_retention_cohorts.sql')
ret.pivot(index='cohort_week', columns='week_n', values='retention_pct').round(0)

week_n,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_week,,,,,,,,,,,,,
2026-01-05,35.0,22.0,17.0,12.0,9.0,7.0,4.0,3.0,2.0,1.0,1.0,1.0,0.0
2026-01-12,36.0,22.0,16.0,12.0,10.0,6.0,4.0,4.0,2.0,2.0,1.0,0.0,1.0
2026-01-19,37.0,23.0,17.0,12.0,8.0,6.0,4.0,3.0,2.0,2.0,1.0,1.0,0.0
2026-01-26,36.0,23.0,17.0,12.0,8.0,7.0,5.0,3.0,3.0,1.0,1.0,1.0,0.0
2026-02-02,38.0,23.0,19.0,14.0,12.0,8.0,5.0,4.0,3.0,2.0,1.0,0.0,0.0
2026-02-09,38.0,25.0,17.0,13.0,9.0,7.0,4.0,3.0,2.0,2.0,0.0,1.0,0.0
2026-02-16,43.0,29.0,22.0,15.0,12.0,9.0,6.0,5.0,3.0,2.0,1.0,1.0,0.0
2026-02-23,40.0,25.0,18.0,15.0,11.0,7.0,6.0,3.0,2.0,2.0,1.0,0.0,0.0
2026-03-02,39.0,24.0,20.0,14.0,10.0,6.0,5.0,3.0,2.0,2.0,1.0,0.0,NaN


## Figures
Run `python src/analysis.py` to regenerate every chart in `reports/figures/` and the written summary in `reports/findings.md`.